# Assignment 4: Image Classification

In this assignment, you will classify images using the [**Darknet framework**](https://darknetcv.ai/api/). You will use OpenCV library and the [**DNN module**](https://docs.opencv.org/4.x/d2/d58/tutorial_table_of_content_dnn.html), which are very fast due to the optimizations inherent in the OpenCV library.

**Step 1: Learning- Classifying one image**

1. Familiarze yourself with popular pre-trained models
2. Learn how to load and apply an image classification model
3. Experiment with the model using additional image examples

**Step 2: Assignment- Classifying Several Images**


## Step 1: Learning - Background

### **Popular pre-trained models**

The OpenCV supports many [**state-of-the-art models**](https://github.com/opencv/opencv/wiki/Deep-Learning-in-OpenCV) including YOLO, TensorFlow, PyTorch, and Caffee. [**This github repo**](https://github.com/opencv/opencv/tree/master/samples/dnn) contains several sample code for several models.

### **Frameworks**

Some popular deep learning frameworks:

*  **Caffe**

You can use the following function to read a network model stored in Caffe framework's format **`readNetFromCaffe()`**. More info here [**`cv2.dnn.readNetFromCaffe()`**](https://docs.opencv.org/4.5.2/d6/d0f/group__dnn.html#ga29d0ea5e52b1d1a6c2681e3f7d68473a)

*  **TensorFlow**

You can use the following function to read a network model stored in [TensorFlow](https://www.tensorflow.org/) framework's format:
**`readNetFromTensorflow()`**. More info here [**`cv2.dnn.readNetFromTensorflow()`**](https://docs.opencv.org/4.5.2/d6/d0f/group__dnn.html#gad820b280978d06773234ba6841e77e8d)

*  **Torch and PyTorch**

For loading Torch model files, we need the file containing the pre-trained weights. Generally, this file has a `.t7` or `.net` extension.

You can use the following function to read a network model stored in [Torch7](http://torch.ch/) framework's format **`readNetFromTorch()`**. More info here [**`cv2.dnn.readNetFromTorch()`**](https://docs.opencv.org/4.5.2/d6/d0f/group__dnn.html#ga65a1da76cb7d6852bdf7abbd96f19084)

*  **Darknet**

You can use the following function to read a network model stored in [Darknet](https://pjreddie.com/darknet/) model files: **`readNetFromDarknet()`**. More info here [**`cv2.dnn.readNetFromDarknet()`**](https://docs.opencv.org/3.4/d6/d0f/group__dnn.html#gafde362956af949cce087f3f25c6aff0d)

*  **Reading any DNN model**

You can use the following function to read deep learning network in any supported format and returns a `Net` object **`readNet()`**. More info here [**`cv2.dnn.readNet()`**](https://docs.opencv.org/4.5.2/d6/d0f/group__dnn.html#ga3b34fe7a29494a6a4295c169a7d32422)

## Image Classification using DNN

For this assignment, you will implement one of the most common computer vision problems using the DNN module. You will use the [**DenseNet121**](https://huggingface.co/qualcomm/DenseNet-121) deep neural network model for classifying images into 1000 classes of [*ImageNet*](https://www.google.com/url?sa=t&rct=j&q=&esrc=s&source=web&cd=&cad=rja&uact=8&ved=2ahUKEwjasLG-yKbyAhW3IbcAHRW1BZkQFnoECAwQAw&url=https%3A%2F%2Fwww.image-net.org%2F&usg=AOvVaw07SoUbpxRQk8ZNEiJoLqt5) dataset.

* The model DenseNet-121 full python code can be found [*here*](https://github.com/pytorch/vision/blob/main/torchvision/models/densenet.py)

* Pretrained DenseNet Models on ImageNet [*DenseNet-Caffee*](https://github.com/shicai/DenseNet-Caffe)


To classify an image, you will go through the following process:

1. Read the class labels from a .txt file – This file contains the names of all the objects the model can recognize.
2. Load the pre-trained neural network using `readNet` function to import the model from disk.
3. Import the input image you want to classify.
4. Convert the image into a blob using `blobFromImage` function.
5. Use the function `setInput` to feed the blob into the network.
6. Run the model to get predictions (`forward`)


In [ ]:
#Import the libraries
import cv2
import numpy as np
import glob
import matplotlib.pyplot as plt
%matplotlib inline

# 1. Read the ImageNet class names from the input folder (classification_classes_ILSVRC2012.txt')
with open('input/classification_classes_ILSVRC2012.txt', 'r') as f:
    image_net_names = f.read().split('\n')

# Save all available classification labels, excluding the empty line at the end.
class_names = image_net_names[:-1]

# Verify the size, and inspect one of the classes by name.
print(len(class_names), class_names[100])

In [ ]:
# 2. Loading the Classification model (caffe) using readNet function.
config_file = 'models/DenseNet_121.prototxt'
model_file = 'models/DenseNet_121.caffemodel'

model = cv2.dnn.readNet(model=model_file, config=config_file, framework='Caffe')

In [ ]:
# 3. Import the image you want to classify.
tiger_img = cv2.imread('input/image1.jpg')
plt.figure(figsize=[10, 10])
plt.imshow(tiger_img[:, :, ::-1]);

4. Convert the image into a blob.

The blob speficications are given at the top of in `DenseNet_121.prototxt`:
```
name: "DENSENET_121"
input: "data"
input_dim: 1  -- Number of images per blob
input_dim: 3   --- Number of channels per image (full color, no alpha)
input_dim: 224   ---(height)
input_dim: 224    ---(width)
```

The mean and scalefactor to use also come from the model itself ([see here](https://github.com/shicai/DenseNet-Caffe)).

In [ ]:
# 4. Convert the image into a blob. The blob specifications are found in DenseNet_121.prototxt.
blob = cv2.dnn.blobFromImage(
    image=tiger_img, scalefactor=0.017, size=(224, 224), mean=(104, 117, 123), swapRB=False, crop=False)

In [ ]:
# 5. Set the input blob for the neural network.
model.setInput(blob)

In [ ]:
# 6. Forward the blob through the network.
outputs = model.forward()
final_outputs = outputs[0]

In [ ]:
# Make all the outputs 1D, where each represents likihood of matching one of the 1K classification groups.
final_outputs = final_outputs.reshape(1000, 1)

# Get the class label index with the max confidence.
label_id = np.argmax(final_outputs)

# Convert score to probabilities for all matches.
probs = np.exp(final_outputs) / np.sum(np.exp(final_outputs))

print(probs[:10])
print("Max probability:", np.max(probs))

In [ ]:
# Get the final highest probability
final_prob = np.max(probs) * 100.0

# Map the max confidence to the class label names.
out_name = class_names[label_id]
out_text = f"{out_name}, {final_prob:.3f}%"

# Display the image, best matched classification, and confidence.
plt.imshow(tiger_img[:, :, ::-1])
plt.title(out_text)
plt.xticks([]), plt.yticks([])
plt.show() 

# Step 2- Assignment: classifying several images

Now that you have successfully implemented Image classification workflow, it's time to apply the model to multiple examples and create a reusable function for batch processing.

We can now give a try at applying the model to more examples. To more neatly use our workflow, we have defined the prios steps into the below reusable function.
Now let us use this function to classify a whole folder of images. Feel free to drop in some of your own jpg images into the `input` folder if you want to see what the model comes up with.


In [ ]:
# 1. Create a function containing all 6 steps to classify an image
#YOUR CODE HERE

# 2. Test Your Implementation
# Use the function you created to classify all the images from the folder "input".
# YOUR CODE HERE

#3. Add more random images to the folder and see how the classification performs.
# YOUR CODE HERE


Reflex and answer the following:

1) What are the advantages of creating a reusable function for the object detection workflow compared to running the steps individually each time?
2) How does batch processing multiple images help you better understand the model's strengths and limitations?
3) Compare the detection results across your test images. Which types of objects were detected most accurately? Which were missed or misclassified?
4) How did image quality, lighting conditions, and object size affect the detection performance?
5) What patterns do you notice in the confidence scores across different images and object types?
6) Based on your results, what real-world applications would this object detection model be suitable for? What limitations would you need to consider?
7) If you were to improve this system for production use, what modifications would you make to the workflow or model parameters?
8) How might you extend this batch processing approach to handle video files or real-time camera input?
